# MyHeritage low-pass WGS: exploratory population-genetics notebook

**Purpose:** local, reproducible exploration of a MyHeritage CRAM file, including alignment QC, variant-call summaries, population PCA and selected exploratory statistics.

**Important:** MyHeritage describes its WGS CRAM as approximately 2× low-pass sequencing. This notebook does not treat unobserved bases as reference alleles and does not perform clinical interpretation. Genotype calling/imputation is an external, validated bioinformatics stage. Results depend on the reference assembly, sequencing characteristics, caller, imputation panel and sample composition.

## Workflow

1. Validate CRAM, index and reference assembly.
2. Summarize alignment and coverage.
3. Generate a low-pass genotype likelihood/imputation dataset using GLIMPSE2 (outside this notebook).
4. Harmonize your sample with a compatible reference panel (e.g. 1000 Genomes Phase 3 on GRCh37).
5. Compute reference PCA and project the sample.
6. Inspect genotype/variant statistics and sensitivity.

**Do not upload your CRAM, VCF, BCF, PLINK files or sample identifiers to public notebooks or repositories.**

## 0. Project configuration

Use a local Linux/WSL2 environment. Keep the original CRAM read-only.

Environment: `my_analysis` from `environment.yml`; it includes the notebook's packages (seaborn, pysam, samtools, ipykernel) and GLIMPSE2.

```bash
mamba env create -f environment.yml
conda activate my_analysis
```

Pin exact tool versions after a successful test run; save `conda list --explicit` and tool version outputs in your project.

All paths come from `configs/config.yaml` (`cram`, `reference_fasta`, `ref_dir`, `out_dir`), the same file the pipeline scripts read:

```text
D:\My_DNA  (/mnt/d/My_DNA, ref_dir)
  MyHeritage_full_sequencing.cram (+ .crai)
  reference/Homo_sapiens_assembly19.fasta (+ .fai, .dict)
  1kg/                      1000 Genomes Phase 3 panel and VCFs
~/projects/my_dna
  outputs/                  everything the pipeline and this notebook read/write (git-ignored)
  scripts/genome/cram_qc.sh        writes outputs/idxstats.tsv, coverage.tsv, chr1.depth_hist.tsv
```

**Clear notebook outputs before committing** (outputs contain results derived from your genome): `jupyter nbconvert --clear-output --inplace myheritage_comprehensive_exploratory_analysis.ipynb`.

In [ ]:
from pathlib import Path
import sys, platform, subprocess, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
               if (p / "scripts" / "common.py").exists())   # works from notebooks/ too
sys.path.insert(0, str(PROJECT / "scripts"))
from common import AUTOSOMES, load_config, load_panel  # shared with the pipeline scripts

# All paths come from configs/config.yaml, the same as the pipeline uses
cfg = load_config()
OUT = cfg["out_dir"]                      # outputs/, shared with the pipeline
CRAM = cfg["cram"]
CRAI = Path(f"{CRAM}.crai")
REFERENCE_FASTA = cfg["reference_fasta"]  # Broad b37 (contigs 1..22, X, Y, MT, NC_007605), matches the CRAM header

RANDOM_SEED = 2026
sns.set_theme(context="notebook", style="whitegrid")

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Project:", PROJECT)
print("Outputs:", OUT)
print("CRAM exists:", CRAM.exists())
print("Index exists:", CRAI.exists())
print("Reference FASTA exists:", REFERENCE_FASTA.exists())

## 1. Validate the CRAM and identify its reference assembly

CRAM decoding requires the correct reference sequence. Do not infer GRCh37/GRCh38 from a filename. Confirm the assembly from MyHeritage documentation/support or the file's provenance, then obtain the exact FASTA and index it.

In [ ]:
import pysam

assert CRAM.exists(), f"CRAM not found: {CRAM}"
assert REFERENCE_FASTA.exists(), f"Reference FASTA not found: {REFERENCE_FASTA}"
# scripts/genome/cram_qc.sh builds the FASTA index and CRAI, so the CRAM opens without index warnings.
missing = [str(p) for p in (Path(f"{REFERENCE_FASTA}.fai"), CRAI) if not p.exists()]
assert not missing, f"Missing index file(s) {missing}: run `bash scripts/genome/cram_qc.sh` first."

# Check CRAM integrity (requires samtools executable).
subprocess.run(["samtools", "quickcheck", "-v", str(CRAM)], check=True)

# Inspect header without printing potentially identifying sample/read-group fields.
with pysam.AlignmentFile(str(CRAM), "rc", reference_filename=str(REFERENCE_FASTA)) as bam:
    header = bam.header.to_dict()
    sq = header.get("SQ", [])
    print("Number of reference sequences:", len(sq))
    print("First reference sequences:")
    for item in sq[:12]:
        print({k: item.get(k) for k in ("SN", "LN", "AS", "M5") if k in item})
    print("Program records:", [
        {k: v for k, v in x.items() if k not in ("CL",)}
        for x in header.get("PG", [])
    ])

print("CRAM validation completed.")

### External command-line QC

Run `bash scripts/genome/cram_qc.sh` in a terminal first. It builds the FASTA and CRAM indexes and writes `idxstats.tsv`, `coverage.tsv` and `chr1.depth_hist.tsv` to `outputs/`, taking paths from the config.

Optional extra checks, not used below (run from the project root; the first line loads `$CRAM`, `$FASTA` and `$OUT` from the config):

```bash
eval "$(python scripts/common.py --shell)"
samtools flagstat --reference "$FASTA" "$CRAM" > "$OUT/flagstat.txt"
samtools stats --reference "$FASTA" "$CRAM" > "$OUT/samtools_stats.txt"
```

Keep the full header private because read-group metadata may identify the sample or laboratory.

In [ ]:
# Parse idxstats output and plot mapped/unmapped read counts by chromosome.
idxstats_file = OUT / "idxstats.tsv"
if idxstats_file.exists():
    idx = pd.read_csv(
        idxstats_file, sep="\t", header=None,
        names=["sequence", "length", "mapped", "unmapped"],
        dtype={"sequence": str}
    )
    display(idx.head())
    # Plot chromosomes in natural order (1..22, X, Y); summarise MT, unplaced GL contigs, EBV and '*' separately.
    main = [str(c) for c in range(1, 23)] + ["X", "Y", "chrX", "chrY"] + [f"chr{c}" for c in range(1, 23)]
    plot_idx = idx.set_index("sequence").reindex([c for c in main if c in set(idx["sequence"])])
    other = idx[~idx["sequence"].isin(main)]
    print(f"Other sequences ({len(other)}): mapped={other['mapped'].sum():,}, unmapped={other['unmapped'].sum():,}")
    ax = plot_idx[["mapped", "unmapped"]].plot(kind="bar", stacked=True, figsize=(14, 4))
    ax.set_ylabel("Read segments")
    ax.set_xlabel("Chromosome")
    plt.tight_layout()
    plt.show()
else:
    print("Run `bash scripts/genome/cram_qc.sh` first; expected:", idxstats_file)

## 2. Coverage assessment

At low-pass depth, mean coverage alone is insufficient. Inspect the distribution of per-base depth, including zero coverage and high-depth tails.

`scripts/genome/cram_qc.sh` produces both inputs: `samtools coverage` for the per-reference summary, and a chr1 depth histogram (`samtools depth -a -r 1`, counted on the fly into a few-KB table of depth → number of bases instead of a 3.4 GB per-base file). This CRAM names chromosomes `1, 2, …`, not `chr1`.

In [ ]:
coverage_file = OUT / "coverage.tsv"
if coverage_file.exists() and coverage_file.stat().st_size > 0:
    # The header line starts with '#rname', so read it as the header and strip the '#'.
    cov = pd.read_csv(coverage_file, sep="\t")
    cov.columns = [c.lstrip("#") for c in cov.columns]
    display(cov.head())
    # samtools coverage column names vary slightly by version; inspect before plotting.
    print("Coverage columns:", cov.columns.tolist())
elif coverage_file.exists():
    print("coverage.tsv is empty: samtools coverage failed or was interrupted; re-run scripts/genome/cram_qc.sh.")
else:
    print("Run `bash scripts/genome/cram_qc.sh` first; expected:", coverage_file)

In [ ]:
hist_file = OUT / "chr1.depth_hist.tsv"
if hist_file.exists():
    # Written by scripts/genome/cram_qc.sh: read depth -> number of chr1 bases at that depth (zero depth included).
    counts = pd.read_csv(hist_file, sep="\t", header=None, names=["depth", "bases"],
                         index_col="depth")["bases"].sort_index()

    n = counts.sum()
    cum = counts.cumsum() / n
    pct = {f"{int(q*100)}%": int(cum.index[np.searchsorted(cum.to_numpy(), q)])
           for q in (.1, .25, .5, .75, .9, .99)}
    summary = pd.Series({"positions": n,
                         "mean": (counts.index.to_numpy() * counts.to_numpy()).sum() / n,
                         "zero-depth fraction": counts.get(0, 0) / n,
                         "max": counts.index.max(), **pct})
    display(summary)

    hist = counts.groupby(np.minimum(counts.index, 15)).sum()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(hist.index, hist.to_numpy(), width=0.9)
    ax.set(xlabel="Read depth (values >15 grouped at 15)", ylabel="Bases",
           title="Chromosome 1 depth distribution")
    plt.tight_layout()
    plt.show()
else:
    print("Run `bash scripts/genome/cram_qc.sh` first; expected:", hist_file)

## 3. Low-pass genotype calling and imputation (external stage)

**Do not use naive hard calls from a 2× CRAM as the main genotype matrix.** Low-pass data have many zero-read sites and uncertain heterozygous calls. Use genotype likelihoods and a validated reference-based imputation workflow.

GLIMPSE2 is designed for low-coverage WGS and produces GT, dosage (DS), genotype probabilities (GP) and imputation information. Follow its official tutorial for:
- compatible FASTA and genetic map;
- compatible phased reference panel;
- chunk definition and reference splitting;
- CRAM input and sample list;
- per-chunk phasing/imputation and ligation.

Official docs: https://odelaneau.github.io/GLIMPSE/

**Assembly:** this CRAM is GRCh37 (Broad b37, contigs `1..22`, `X`, `Y`, `MT`), matching 1000 Genomes Phase 3, so no liftover is needed. Use GRCh37 genetic maps and b37 contig naming (no `chr` prefix) throughout.

**Base qualities:** `samtools coverage` reports `meanbaseq = 255` for every contig, i.e. per-base qualities are not stored in this CRAM (Ultima Genomics-style pipeline). Genotype likelihoods depend on base quality, so check how your GLIMPSE2 version treats missing qualities (and its `--mapq`/`--baseq` filters) on a small region before running genome-wide, and validate against the MyHeritage array genotypes (`MyHeritage_raw_dna_data.csv`) where they overlap.

Example outputs expected for the downstream notebook:
- `outputs/sample.imputed.vcf.gz` (or BCF), with DS/GP/INFO fields;
- a matched reference panel genotype dataset;
- a sample metadata panel.

This notebook intentionally does not provide a one-line GLIMPSE2 command: its commands depend on the exact reference panel, map, assembly, contig naming, chunk files and GLIMPSE2 version. Copy the complete command sequence from the matching GLIMPSE2 tutorial and record all parameters.

In [ ]:
# Inspect an imputed VCF/BCF after GLIMPSE2.
import itertools
IMPUTED_VCF = OUT / "sample.imputed.vcf.gz"

if IMPUTED_VCF.exists():
    if not (Path(str(IMPUTED_VCF) + ".tbi").exists() or Path(str(IMPUTED_VCF) + ".csi").exists()):
        subprocess.run(["bcftools", "index", "-t", str(IMPUTED_VCF)], check=True)
    # Stream and stop after 5 records; a genome-wide VCF is far too large to capture in memory.
    with subprocess.Popen(
        ["bcftools", "query", "-f", "%CHROM\t%POS\t%ID\t%REF\t%ALT[\t%GT\t%DS\t%GP]\n", str(IMPUTED_VCF)],
        stdout=subprocess.PIPE, text=True
    ) as proc:
        for line in itertools.islice(proc.stdout, 5):
            print(line.rstrip("\n"))
        proc.terminate()
else:
    print("Imputed VCF not present yet:", IMPUTED_VCF)

## 4. Variant-level QC and descriptive statistics

The following code uses `bcftools` to stream an imputed VCF into a compact table (a genome-wide imputed VCF has tens of millions of records, too many for a per-record Python loop). It summarizes variant counts and, when available, the imputation INFO score. These are exploratory summaries, not a clinical variant report.

For population analysis, restrict to high-quality, common, biallelic autosomal SNPs, harmonize alleles and apply LD pruning. Rare variants and indels require separate handling.

In [ ]:
def summarize_vcf(vcf_path):
    """Biallelic autosomal SNPs with their INFO score, streamed via bcftools (no ID column, to save memory)."""
    view = subprocess.Popen(
        ["bcftools", "view", "-v", "snps", "-m2", "-M2", "-Ou", str(vcf_path)],
        stdout=subprocess.PIPE
    )
    query = subprocess.Popen(
        ["bcftools", "query", "-f", "%CHROM\t%POS\t%REF\t%ALT\t%INFO/INFO\n", "-"],
        stdin=view.stdout, stdout=subprocess.PIPE
    )
    view.stdout.close()
    df = pd.read_csv(
        query.stdout, sep="\t", header=None, names=["chrom", "pos", "ref", "alt", "info"],
        dtype={"chrom": "category", "pos": "int32", "ref": "category", "alt": "category"},
        na_values="."
    )
    for p in (query, view):
        if p.wait() != 0:
            raise RuntimeError(f"bcftools failed: {p.args}")
    df["info"] = pd.to_numeric(df["info"], errors="coerce").astype("float32")
    return df[df["chrom"].isin(AUTOSOMES)]  # AUTOSOMES from common.py: b37 names 1..22.reset_index(drop=True)

if IMPUTED_VCF.exists():
    variants = summarize_vcf(IMPUTED_VCF)
    print("Biallelic autosomal SNPs:", len(variants))
    display(variants.head())
    if variants["info"].notna().any():
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(variants["info"].dropna(), bins=30, ax=ax)
        ax.set(title="Imputation INFO distribution", xlabel="INFO", ylabel="Variants")
        plt.tight_layout()
        plt.show()
else:
    print("Complete imputation before variant QC.")

## 5. Prepare the reference population panel

A useful starting point is 1000 Genomes Phase 3, whose collection is on GRCh37. It contains diverse global populations, including IBS (Iberian individuals from Spain), GBR, TSI, FIN and CEU. The project portal supplies sample panels and per-chromosome genotype files.

Reference:
https://www.internationalgenome.org/data-portal/data-collection/phase3

Use a matching phased reference panel for GLIMPSE2, and use the same assembly, variant representation and allele coding for PCA. Keep the population panel and sample IDs as supplied. For an exploratory European comparison, include IBS, GBR, TSI, FIN and CEU; retain a global panel for context, but do not interpret continental axes as precise national ancestry.

In [ ]:
if cfg["panel"].exists():
    panel = load_panel(cfg)  # same loader as the pipeline (drops the empty trailing-tab columns)
    display(panel.head())
    print(panel.columns.tolist())
    print(panel.groupby("pop").size().sort_values(ascending=False).head(20))
else:
    print("Run `bash scripts/setup/download_1kg.sh` to download the sample panel:", cfg["panel"])

## 6. PCA with the 1000 Genomes reference

`scripts/ethnicity/ancestry.sh` fits this PCA (PLINK 2 `--pca 10` on LD-pruned SNPs with MAF ≥ 5%) on the 2,504 1000 Genomes samples together with you. You are 1 of 2,505 samples and your genotype matrix is complete (imputed MyHeritage calls), so your effect on the axes is negligible and the result is close to a projection. The protocol below is what a strict reference-fit-then-project analysis adds.

Avoid fitting PCA jointly to a high-quality reference panel and one low-pass individual with different missingness/uncertainty. Fit PCA on the reference individuals, LD-pruned variants and project the target sample using the reference allele frequencies and loadings.

A robust protocol:
1. Select high-quality, unrelated reference samples.
2. Restrict to autosomal, biallelic SNPs with consistent alleles.
3. Apply missingness and allele-frequency filters.
4. LD-prune the reference SNPs.
5. Fit reference PCA.
6. Project the target using reference loadings/frequencies.
7. Validate projection by holding out reference individuals and projecting them as pseudo-targets.

PLINK 2 docs: https://www.cog-genomics.org/plink/2.0/str

Do not use the illustrative commands from another pipeline without checking the exact PLINK 2 version and output column schema. PLINK projection score scaling must be checked against held-out reference samples.

### Pipeline PCA files

The cells below read the pipeline's output directly and reuse the functions in `scripts/ethnicity/plot_ancestry.py`, so the notebook and the script report the same numbers:
- `outputs/ancestry.eigenvec`: FID, IID, PC1–PC10 for the reference samples and you (IID = `sample_id` in the config);
- `outputs/ancestry.eigenval`: eigenvalues, used to weight PCs in distances;
- the 1000 Genomes sample panel, for population and superpopulation labels.

Run `bash scripts/genome/merge_1kg.sh` and `bash scripts/ethnicity/ancestry.sh` first. Their LD-pruning setting (`--indep-pairwise 200 50 0.2`) is a common default, not a universal optimum; see the sensitivity list in section 9.

In [ ]:
from ethnicity import plot_ancestry as anc

HAVE_PCA = (OUT / "ancestry.eigenvec").exists()
if HAVE_PCA:
    pcs, pc_cols = anc.load_pcs(cfg)           # eigenvectors joined to the 1000G panel
    me = pcs[pcs["IID"] == cfg["sample_id"]].iloc[0]
    ref = pcs.dropna(subset=["pop"])           # reference samples only
    eigval = np.loadtxt(OUT / "ancestry.eigenval")
    print(f"{len(ref)} reference samples, {len(pc_cols)} PCs")
    display(ref.head())
else:
    print("Run `bash scripts/ethnicity/ancestry.sh` first; expected:", OUT / "ancestry.eigenvec")

In [ ]:
if HAVE_PCA:
    european = ["IBS", "GBR", "TSI", "FIN", "CEU"]
    sub = ref[ref["pop"].isin(european)]

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(data=sub, x="PC1", y="PC2", hue="pop", alpha=.65, s=35, ax=ax)
    ax.scatter(me["PC1"], me["PC2"], marker="*", s=250, edgecolor="black", color="black", label="You")
    ax.legend()  # rebuild so you appear alongside the populations
    ax.set_title("European 1000 Genomes populations on the global PCA, with you")
    plt.tight_layout()
    plt.show()
else:
    print("PCA inputs not yet available.")

## 7. Genetic distances and population summaries

A simple descriptive alternative to interpreting only a PCA plot is to calculate distances in the same projected PC space. This is not an ancestry percentage and depends on which PCs, scaling and reference individuals are used.

Use several PCs and report sensitivity to the number of PCs. Prefer population centroids and within-population dispersion rather than the nearest single individual.

In [ ]:
if HAVE_PCA:
    # Distance to each population centroid over PC1-10, PCs weighted by sqrt(eigenvalue);
    # the same function scripts/ethnicity/plot_ancestry.py prints, so the two always agree.
    distance_table = anc.nearest_populations(pcs, pc_cols, me, eigval)
    display(distance_table.head(15))
else:
    print("PCA inputs not yet available.")

## 8. Variant annotation (optional)

For selected variants, use Ensembl VEP or another well-documented annotation resource after normalizing and validating variants against the correct assembly. Annotation should include transcript consequences, allele frequencies and evidence sources. Do not interpret an annotation label as a diagnosis.

VEP: https://www.ensembl.org/info/docs/tools/vep/index.html

For ancestry exploration, functional annotation is secondary. Avoid running health-risk interpretation on a consumer low-pass file.

## 9. Statistical robustness and limitations

Minimum sensitivity analyses:
- Compare PCA after multiple LD-pruning thresholds.
- Compare global versus European-only reference PCA.
- Vary imputation INFO thresholds (for example 0.4, 0.6, 0.8), while treating these as sensitivity settings rather than universal cutoffs.
- Compare dosage-based results with high-confidence genotype calls.
- Hold out reference individuals and assess projection accuracy.
- Bootstrap reference individuals within populations to quantify centroid uncertainty.
- Repeat distances using different numbers of PCs.

Interpretation limitations:
- PCA is conditional on the reference panel and variants retained.
- IBS is a sample of Iberian individuals, not a complete representation of Spain or the Basque Country.
- Genetic similarity does not establish nationality, ethnicity, language, cultural identity or a specific ancestor.
- Low-pass imputation quality varies by allele frequency, ancestry match and sequencing characteristics.
- A single individual cannot support reliable population-level inference.
- This is exploratory research, not medical, legal or forensic analysis.

In [ ]:
# Bootstrap population centroids in PCA space (illustrative uncertainty analysis).
# Uses ref/me/eigval from section 6 and the same PC weighting as nearest_populations.
if HAVE_PCA:
    rng = np.random.default_rng(RANDOM_SEED)
    boot_pcs = pc_cols[:5]
    w = np.sqrt(eigval[: len(boot_pcs)])
    target = me[boot_pcs].to_numpy(float)

    records = []
    for pop, group in ref.groupby("pop"):
        if len(group) < 5:
            continue
        x = group[boot_pcs].to_numpy(float)
        idx = rng.integers(0, len(x), size=(1000, len(x)))  # 1000 resamples with replacement
        boot_d = np.sqrt((((x[idx].mean(axis=1) - target) * w) ** 2).sum(axis=1))
        records.append({
            "population": pop,
            "distance_median": np.median(boot_d),
            "distance_low": np.quantile(boot_d, .025),
            "distance_high": np.quantile(boot_d, .975),
            "n_reference": len(group)
        })
    bootstrap_distances = pd.DataFrame(records).sort_values("distance_median")
    display(bootstrap_distances.head(15))
else:
    print("PCA inputs not yet available.")

## 10. Reproducibility and privacy checklist

Before sharing results:
- Record CRAM provenance, genome assembly, FASTA checksum, tool versions, reference-panel release and command-line parameters.
- Keep a copy of the original CRAM unchanged.
- Do not place CRAM/VCF/BCF/PLINK files, sample IDs or raw genotypes in Git or public cloud notebooks.
- Remove identifying metadata from logs and notebook outputs.
- Remember that genetic data can reveal information about biological relatives as well as the tested person.
- Keep intermediate genotype files encrypted and access-controlled.
- Check the current terms and privacy policies before sending data to any third-party service.

MyHeritage states that the downloaded CRAM is the user's responsibility and is no longer protected by the site's security/privacy settings; it also says the file is not intended for medical, health or legal use.

## References
- MyHeritage CRAM download and coverage FAQ: https://www.myheritage.com/help/en/articles/12851869-how-do-i-download-my-raw-dna-data-file-or-cram-file-from-myheritage
- GLIMPSE2 documentation: https://odelaneau.github.io/GLIMPSE/
- 1000 Genomes Phase 3: https://www.internationalgenome.org/data-portal/data-collection/phase3
- PLINK 2: https://www.cog-genomics.org/plink/2.0/
- Ensembl VEP: https://www.ensembl.org/info/docs/tools/vep/index.html

**Notebook status:** This is a reproducible analysis scaffold. It runs end-to-end once `scripts/genome/cram_qc.sh`, `scripts/genome/merge_1kg.sh` and `scripts/ethnicity/ancestry.sh` have been run; sections 3–4 additionally need the GLIMPSE2 imputation output. It intentionally does not fabricate genotype calls or sample coordinates.